In [1]:
# ============================================================
# CELL 1 — NISQA setup
# ============================================================
import sys
import os

NISQA_REPO = "/Users/abey/Documents/NISQA_PROD/model"
if NISQA_REPO not in sys.path:
    sys.path.insert(0, NISQA_REPO)

from nisqa.NISQA_model import nisqaModel
import pandas as pd

print("✅ NISQA imported successfully")

✅ NISQA imported successfully


In [2]:
# ============================================================
# CELL 2 — Paths and thresholds
# ============================================================
# Change BASE_DIR to your project folder
# Reference folder is optional — pipeline works without it

BASE_DIR      = "/Users/abey/Documents/NISQA_PROD"
REFERENCE_DIR = os.path.join(BASE_DIR, "reference")
MODELS_DIR    = os.path.join(BASE_DIR, "models")
NISQA_WEIGHT  = os.path.join(NISQA_REPO, "weights", "nisqa.tar")

# ── thresholds — calibrate with editor later ──
THRESHOLDS = {
    "MOS"           : 3.0,
    "Noisiness"     : 3.5,
    "Discontinuity" : 3.5,
    "Coloration"    : 3.0,
    "Loudness"      : 3.0,
}

# ── delta thresholds ──
DELTA_THRESHOLDS = {
    "MOS"           : -0.5,
    "Noisiness"     : -0.5,
    "Discontinuity" : -0.5,
    "Coloration"    : -0.5,
    "Loudness"      : -0.6,
}

print("✅ Paths and thresholds set")

✅ Paths and thresholds set


In [3]:
# ============================================================
# CELL 3 — score_single_file function
# ============================================================
def score_single_file(audio_path):
    args = {
        "mode"            : "predict_file",
        "pretrained_model": NISQA_WEIGHT,
        "deg"             : audio_path,
        "data_dir"        : None,
        "output_dir"      : None,
        "csv_file"        : None,
        "csv_deg"         : None,
        "tr_bs_val"       : 1,
        "tr_num_workers"  : 0,
        "ms_channel"      : None,
    }
    nisqa   = nisqaModel(args)
    df_pred = nisqa.predict()
    row     = df_pred.iloc[0]

    return {
        "MOS"           : round(float(row["mos_pred"]),  3),
        "Noisiness"     : round(float(row["noi_pred"]),  3),
        "Discontinuity" : round(float(row["dis_pred"]),  3),
        "Coloration"    : round(float(row["col_pred"]),  3),
        "Loudness"      : round(float(row["loud_pred"]), 3),
    }

print("✅ score_single_file defined")

✅ score_single_file defined


In [4]:
# ============================================================
# CELL 4 — Startup validation
# ============================================================

# ── check models folder ──
if not os.path.exists(MODELS_DIR):
    raise FileNotFoundError(f"Models folder not found: {MODELS_DIR}")

# ── discover model folders ──
model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

# ── discover wav files per model ──
model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([
        f for f in os.listdir(model_path)
        if f.endswith(".wav")
    ])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

# ── validate all models have identical filenames ──
reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

# ── check reference folder — optional ──
reference_available = os.path.exists(REFERENCE_DIR)
if reference_available:
    ref_files = sorted([
        f for f in os.listdir(REFERENCE_DIR)
        if f.endswith(".wav")
    ])
    print(f"✅ Reference folder found: {len(ref_files)} files")
else:
    print("⚠️  No reference folder found — absolute thresholds only")

# ── summary ──
sample_names = model_samples[model_folders[0]]
total        = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")
print(f"Reference: {'available' if reference_available else 'not available'}")

✅ Models found: ['m1', 'm2']
   m1: 2 samples
   m2: 2 samples
✅ All models have identical filenames
✅ Reference folder found: 2 files

Ready: 2 models × 2 samples = 4 evaluations
Reference: available


In [5]:
# ============================================================
# CELL 5 — Main evaluation loop
# ============================================================

def get_absolute_pass(scores):
    # returns True if all sub-scores meet threshold
    return all(scores[k] >= THRESHOLDS[k] for k in THRESHOLDS)

def get_delta_pass(deltas):
    # returns True if all deltas are within acceptable range
    return all(deltas[k] >= DELTA_THRESHOLDS[k] for k in DELTA_THRESHOLDS)

def get_primary_failure_with_delta(deltas):
    # sub-score with largest negative delta
    return min(deltas, key=deltas.get)

def get_primary_failure_without_delta(scores):
    # sub-score furthest below its threshold
    distances = {k: scores[k] - THRESHOLDS[k] for k in THRESHOLDS}
    return min(distances, key=distances.get)

# ── main loop ──
results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        tts_path    = os.path.join(MODELS_DIR, model, wav_file)

        print(f"\n  Sample: {sample_name}")

        # ── score TTS audio ──
        tts_scores = score_single_file(tts_path)
        print(f"  TTS    → MOS: {tts_scores['MOS']} | Noi: {tts_scores['Noisiness']} | Dis: {tts_scores['Discontinuity']} | Col: {tts_scores['Coloration']} | Lou: {tts_scores['Loudness']}")

        # ── absolute pass/fail ──
        absolute_pass = get_absolute_pass(tts_scores)

        # ── reference scoring ──
        deltas       = None
        delta_pass   = None
        ref_scores   = None
        ref_flag     = None

        if reference_available:
            ref_path = os.path.join(REFERENCE_DIR, wav_file)

            if not os.path.exists(ref_path):
                ref_flag = "NO_REF"
                print(f"  ⚠️  No reference file found for {wav_file}")

            else:
                ref_scores = score_single_file(ref_path)
                print(f"  REF    → MOS: {ref_scores['MOS']} | Noi: {ref_scores['Noisiness']} | Dis: {ref_scores['Discontinuity']} | Col: {ref_scores['Coloration']} | Lou: {ref_scores['Loudness']}")

                if ref_scores["MOS"] < 3.0:
                    ref_flag = "REF_QUALITY"
                    print(f"  ⚠️  Reference MOS below 3.0 — skipping delta")

                else:
                    # compute deltas
                    deltas = {
                        k: round(tts_scores[k] - ref_scores[k], 3)
                        for k in tts_scores
                    }
                    delta_pass = get_delta_pass(deltas)
                    print(f"  DELTA  → MOS: {deltas['MOS']} | Noi: {deltas['Noisiness']} | Dis: {deltas['Discontinuity']} | Col: {deltas['Coloration']} | Lou: {deltas['Loudness']}")
        else:
            ref_flag = "NO_REF"

        # ── hybrid pass logic ──
        if deltas is not None:
            if absolute_pass and delta_pass:
                final_result    = "✅ PASS"
                primary_failure = "—"
            elif absolute_pass and not delta_pass:
                final_result    = "✅ PASS"
                primary_failure = get_primary_failure_with_delta(deltas)
            elif not absolute_pass and delta_pass:
                final_result    = "⚠️ REVIEW"
                primary_failure = get_primary_failure_with_delta(deltas)
            else:
                final_result    = "❌ FAIL"
                primary_failure = get_primary_failure_with_delta(deltas)
        else:
            # no delta — absolute only
            if absolute_pass:
                final_result    = "✅ PASS"
                primary_failure = "—"
            else:
                final_result    = "❌ FAIL"
                primary_failure = get_primary_failure_without_delta(tts_scores)

        print(f"  Result : {final_result} | Primary failure: {primary_failure} | Flag: {ref_flag or '—'}")

        # ── store result ──
        row = {
            "Model"           : model,
            "Sample"          : sample_name,
            # TTS scores
            "MOS"             : tts_scores["MOS"],
            "Noisiness"       : tts_scores["Noisiness"],
            "Discontinuity"   : tts_scores["Discontinuity"],
            "Coloration"      : tts_scores["Coloration"],
            "Loudness"        : tts_scores["Loudness"],
            # deltas
            "ΔMOS"            : deltas["MOS"]           if deltas else None,
            "ΔNoisiness"      : deltas["Noisiness"]     if deltas else None,
            "ΔDiscontinuity"  : deltas["Discontinuity"] if deltas else None,
            "ΔColoration"     : deltas["Coloration"]    if deltas else None,
            "ΔLoudness"       : deltas["Loudness"]      if deltas else None,
            # results
            "Absolute"        : "✅" if absolute_pass else "❌",
            "Final"           : final_result,
            "Primary Failure" : primary_failure,
            "Flag"            : ref_flag or "—",
        }
        results.append(row)

print("\n\nAll evaluations complete.")


Model: m1

  Sample: clean_baseline 2
Device: cpu
Model architecture: NISQA_DIM
Loaded pretrained model from /Users/abey/Documents/NISQA_PROD/model/weights/nisqa.tar
---> Predicting ...


/Users/abey/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                 deg  mos_pred  noi_pred  dis_pred  col_pred  loud_pred
clean_baseline 2.wav  4.289991  4.415084  4.285129  4.008628   4.288787
  TTS    → MOS: 4.29 | Noi: 4.415 | Dis: 4.285 | Col: 4.009 | Lou: 4.289
Device: cpu
Model architecture: NISQA_DIM
Loaded pretrained model from /Users/abey/Documents/NISQA_PROD/model/weights/nisqa.tar
---> Predicting ...
                 deg  mos_pred  noi_pred  dis_pred  col_pred  loud_pred
clean_baseline 2.wav  4.289991  4.415084  4.285129  4.008628   4.288787
  REF    → MOS: 4.29 | Noi: 4.415 | Dis: 4.285 | Col: 4.009 | Lou: 4.289
  DELTA  → MOS: 0.0 | Noi: 0.0 | Dis: 0.0 | Col: 0.0 | Lou: 0.0
  Result : ✅ PASS | Primary failure: — | Flag: —

  Sample: clean_baseline
Device: cpu
Model architecture: NISQA_DIM
Loaded pretrained model from /Users/abey/Documents/NISQA_PROD/model/weights/nisqa.tar
---> Predicting ...
               deg  mos_pred  noi_pred  dis_pred  col_pred  loud_pred
clean_baseline.wav  4.289991  4.415084  4.285129  4.008628   

In [6]:
# ============================================================
# CELL 6 — Results tables and ranking
# ============================================================
df = pd.DataFrame(results)

# ── Table 1 — full per segment results ──
print("\n========== FULL PER-SEGMENT RESULTS ==========")
display_cols = [
    "Model", "Sample",
    "MOS", "Noisiness", "Discontinuity", "Coloration", "Loudness",
    "ΔMOS", "ΔNoisiness", "ΔDiscontinuity", "ΔColoration", "ΔLoudness",
    "Absolute", "Final", "Primary Failure", "Flag"
]
print(df[display_cols].to_string(index=False))

# ── Table 2 — per model summary ──
print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df    = df[df["Model"] == model]
    wer_values  = model_df["MOS"]
    delta_values = model_df["ΔMOS"].dropna()

    pass_count   = (model_df["Final"] == "✅ PASS").sum()
    review_count = (model_df["Final"] == "⚠️ REVIEW").sum()
    fail_count   = (model_df["Final"] == "❌ FAIL").sum()
    total        = len(model_df)

    # most common primary failure mode
    failure_counts = model_df[
        model_df["Primary Failure"] != "—"
    ]["Primary Failure"].value_counts()
    most_common_failure = failure_counts.index[0] if len(failure_counts) > 0 else "—"

    summary_rows.append({
        "Model"                : model,
        "Segments"             : total,
        "Mean MOS"             : round(wer_values.mean(), 3),
        "Median MOS"           : round(wer_values.median(), 3),
        "Mean ΔMOS"            : round(delta_values.mean(), 3) if len(delta_values) > 0 else None,
        "Median ΔMOS"          : round(delta_values.median(), 3) if len(delta_values) > 0 else None,
        "Pass Rate"            : f"{pass_count}/{total}",
        "Review Rate"          : f"{review_count}/{total}",
        "Fail Rate"            : f"{fail_count}/{total}",
        "Top Failure Mode"     : most_common_failure,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Table 3 — model ranking ──
print("\n========== MODEL RANKING ==========")

# extract numeric pass rate for sorting
summary_df["_pass_num"] = summary_df["Pass Rate"].apply(
    lambda x: int(x.split("/")[0])
)

ranking = summary_df.sort_values(
    by=["_pass_num", "Median ΔMOS"],
    ascending=[False, False]
)[[
    "Model", "Mean MOS", "Median MOS",
    "Mean ΔMOS", "Pass Rate", "Fail Rate", "Top Failure Mode"
]]

print(ranking.to_string(index=False))

print("\n========== WHAT TO LOOK FOR ==========")
print("Best model      → highest Pass Rate, least negative Mean ΔMOS")
print("REVIEW segments → absolute fail but delta small — listen before deciding")
print("Top Failure     → tells you which artifact type this model produces most")
print("Flag NO_REF     → no reference available, absolute threshold only")
print("Flag REF_QUALITY→ reference MOS too low to trust delta")


========== FULL PER-SEGMENT RESULTS ==========
Model           Sample  MOS  Noisiness  Discontinuity  Coloration  Loudness  ΔMOS  ΔNoisiness  ΔDiscontinuity  ΔColoration  ΔLoudness Absolute  Final Primary Failure Flag
   m1 clean_baseline 2 4.29      4.415          4.285       4.009     4.289   0.0         0.0             0.0          0.0        0.0        ✅ ✅ PASS               —    —
   m1   clean_baseline 4.29      4.415          4.285       4.009     4.289   0.0         0.0             0.0          0.0        0.0        ✅ ✅ PASS               —    —
   m2 clean_baseline 2 4.29      4.415          4.285       4.009     4.289   0.0         0.0             0.0          0.0        0.0        ✅ ✅ PASS               —    —
   m2   clean_baseline 4.29      4.415          4.285       4.009     4.289   0.0         0.0             0.0          0.0        0.0        ✅ ✅ PASS               —    —

========== MODEL COMPARISON SUMMARY ==========
Model  Segments  Mean MOS  Median MOS  Mean ΔMOS 